In [1]:
from openai import OpenAI
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
import os
from dotenv import load_dotenv

# Load variables from .env file into os.environ
load_dotenv()

# Read the endpoint from the environment
endpoint = os.getenv("OPENAI_ENDPOINT")
if not endpoint:
    raise ValueError("OPENAI_ENDPOINT environment variable is missing or empty.")

# Azure OpenAI
token_provider = get_bearer_token_provider(
    DefaultAzureCredential(), "https://ai.azure.com/.default"
)
openai_client = OpenAI(  
  base_url = endpoint,  
  api_key=token_provider,
)

# Create vector store and upload files

In [2]:
import glob
# The glob module in Python is used to retrieve files or directories that match a specified pattern. 
# It is particularly useful for searching through file systems in a way similar to using shell wildcards.

# Create vector store and upload files
print("Creating vector store and uploading files...")
vector_store = openai_client.vector_stores.create(
    name="travel-brochures"
)
file_streams = [open(f, "rb") for f in glob.glob("brochures/*.pdf")]
if not file_streams:
    raise FileNotFoundError("No PDF files found in the brochures folder.")
file_batch = openai_client.vector_stores.file_batches.upload_and_poll(
     vector_store_id=vector_store.id,
     files=file_streams
)
for f in file_streams:
    f.close()
print(f"Vector store created with {file_batch.file_counts.completed} files.")

Creating vector store and uploading files...
Vector store created with 6 files.


In [42]:
input_text = "What's happening in San Francisco next month?"

# Get a response using tools
response = openai_client.responses.create(
    model="gpt-6-luna",
    instructions = """
    You are a reliable AI assistant equipped with two knowledge tools:
    1. `file_search`: Provides internal/private documents, guides, and domain-specific knowledge.
    2. `web_search`: Accesses live, external information across the internet.

    ### TOOL ROUTING RULES:
    Before responding to any query, determine the required source and invoke the appropriate tool:

    1. USE `file_search` WHEN:
    - The user asks about organization-specific policies, proprietary services, internal guides, or uploaded documentation.
    - The query references private context (e.g., contracts, brochures, company FAQs).

    2. USE `web_search` WHEN:
    - The question requires real-time, current, or recent information (e.g., weather, news, current events, live rates).
    - The user asks about general external topics, places, public entities, or facts not covered in internal documents.
    - An internal search (`file_search`) returns insufficient or no relevant results, but the answer can be found publicly on the web.

    3. USE BOTH TOOLS WHEN:
    - The user asks a hybrid question (e.g., "Compare company package X from our brochure with current entry requirements for destination Y").

    ### CRITICAL CONSTRAINTS:
    - Do NOT guess, assume, or answer from memory if the question requires up-to-date facts or proprietary data.
    - Always execute the appropriate tool call before providing your final answer.
    - Ground all facts in the tool search outputs.
    """,
    input=input_text,
    tools=[
        {
            "type": "file_search",
            "vector_store_ids": [vector_store.id]
        },
        {
            "type": "web_search"
        }
    ]
)

print(response.output_text)

**October 2026 is busy in San Francisco**, especially the first two weekends:

- **Oct. 2–4 — Hardly Strictly Bluegrass:** A free, three-day music festival in Golden Gate Park. No tickets needed. ([hardlystrictlybluegrass.com](https://hardlystrictlybluegrass.com/info-faq-2026/))
- **Oct. 4 — Castro Street Fair:** A neighborhood celebration with local artists, food, music, and dancing, 11 a.m.–6 p.m. ([castrostreetfair.org](https://castrostreetfair.org/))
- **Oct. 4–12 — Fleet Week:** Waterfront events, ship tours, and the Blue Angels air show **Oct. 9–11**. ([fleetweeksf.org](https://fleetweeksf.org/))
- **Oct. 11 — Italian Heritage Parade:** A free North Beach tradition with music, performances, and floats. ([sf.funcheap.com](https://sf.funcheap.com/city-guide/october-street-fairs-festivals/))
- **Oct. 17 — Bay Area Science Festival:** Plus neighborhood fairs that weekend, including Bearrison Street Fair and Potrero Hill’s R&B Festival. ([sf.funcheap.com](https://sf.funcheap.com/city-

When prompted, enter "What's happening in San Francisco next month?".

The response should include information retrieved using the web_search tool. 
* FAILS with gpt-4.1-nano: The brochures provided do not specify specific events or happenings in San Francisco for next month. They mainly offer general information about the city, accommodations, and travel options. For detailed events or happenings, I recommend checking local event calendars or the official San Francisco tourism website.
* SUCCESS only when web_search is the only tool available, even with better instructions.
* SUCCESS with gpt-6-luna (better model)

Try this follow-up question: "What hotels does Margie's Travel offer there?"

The response should include information retrieved using the file_search tool.
* SUCCESS with gpt-4.1-nano: In San Francisco, Margie's Travel offers two main hotel options: The Lombard Hotel, which is within walking distance to popular sights like the Golden Gate Bridge and the Presidio, and The Wharf Hotel, which is located in the vibrant Fisherman's Wharf area and includes a continental breakfast.

When you’re finished, enter quit to exit the program.

Initial basic instructions:
    """
    You are a travel assistant that provides information on travel services available from Margie's Travel.
    Answer questions about services offered by Margie's Travel using the provided travel brochures.
    Search the web for general information about destinations or current travel advice.
    """

# Call a function

In [13]:
import time
from openai import OpenAI

# Function to get the current time
def get_time():
    return f"The time is {time.strftime('%Y-%m-%d %H:%M:%S', time.localtime())}"


model = "gpt-4.1-nano"

function_tools = [
    {
        "type": "function",
        "name": "get_time",
        "description": "Get the current time"
    }
]

# Initialize messages with a system prompt
messages = [
    {"role": "developer", "content": "You are an AI assistant that provides information."},
]

# Loop until the user types 'quit'
while True:
    prompt = input("\nEnter a prompt (or type 'quit' to exit)\n")
    if prompt.lower() == "quit":
        break

    # Append the user prompt to the messages
    messages.append({"role": "user", "content": prompt})

    # Get initial response
    response = openai_client.responses.create(
        model=model,
        input=messages,
        tools=function_tools
    )

    # Append model output to the messages
    messages += response.output

    # Was there a function call?
    for item in response.output:
        if item.type == "function_call" and item.name == "get_time":
            current_time = get_time()
            messages.append({
                "type": "function_call_output",
                "call_id": item.call_id,
                "output": current_time
            })

            # Get a follow up response using the tool output
            response = openai_client.responses.create(
                model=model,
                instructions="Answer only with the tool output.",
                input=messages,
                tools=function_tools
            )

    print(response.output_text)

Hello! How can I assist you today?


In this flow, the model decides when to call get_time, your code runs the function, and the model then returns a grounded final answer. Since the user can enter any prompt, the model must determine when it needs to call the function. If it does, the response to the prompt will include a function call, that the application code must implement before submitting a new prompt with the output from the function for the model to process.

# Code interpreter

In [ ]:
input_text = "What is the square root of 26273?"

# Get a response using tools
response = openai_client.responses.create(
    model="gpt-4.1-nano",
    instructions = """
    You are a reliable AI assistant equipped with one knowledge tools.
    `code_interpreter`: Executes code and returns the output.

    ### TOOL ROUTING RULES:
    Before responding to any query, determine if invoking the appropriate tool is relevant.

    USE `code_interpreter` WHEN:
    - The user asks for the execution of code.
    - The question involves programming tasks that require code execution.
    - The request requires mathematical computation.

    ### CRITICAL CONSTRAINTS:
    - Do NOT guess, assume, or answer from memory if the question requires up-to-date facts or proprietary data.
    - Always execute the appropriate tool call before providing your final answer.
    - Ground all facts in the tool search outputs.
    """,
    input=input_text,
    tools=[
        {
            "type": "code_interpreter",
            "container": {
                "type": "auto"
            }
        }
    ]
)

# Quick Shortcut for Assistant Text
# If you only need the final text output and don't need to manually parse item.content, the SDK provides a helper property:
print(response.output_text)

The square root of 26273 is approximately 162.09.


In [43]:
# Print the full response for debugging
#print(response)

# Iterate through the generated response items
if response.output:
    for item in response.output:
        if item.type == "code_interpreter_call":
            print("\nCode Interpreter Executed:")
            print(f"--- Code ---\n{item.code.strip()}")
            print("\n--- Output ---")
            if item.outputs:
                for out in item.outputs:
                    # Logs and prints are stored under 'logs'
                    if hasattr(out, "logs"):
                        print(out.logs)
                    else:
                        print(out)
            else:
                print("(Outputs omitted by default; pass include=['code_interpreter_call.outputs'] to fetch)")

        elif item.type == "web_search_call":
            print(f"\nWeb Search Query: {getattr(item, 'query', item)}")

        elif item.type == "message":
            # Extract plain text from each content block inside the message
            message_text = "".join(
                content_part.text
                for content_part in item.content
                if content_part.type == "output_text"
            )
            print(f"\nAssistant Response:\n{message_text}")

        else:
            print(f"\nOther item ({item.type}):\n{item}")

else:
    print("No response received.")


Other item (reasoning):
ResponseReasoningItem(id='rs_0410619190c4ecc2006abbb0619f388197a21ae6cae02b3e64', summary=[], type='reasoning', content=[], encrypted_content='gAAAAABqu7Br4bNydpd77cCVyBC-OlexJ4Q-YyIFGEZ4sKVZwCHT5_mrv31GDJFOLPzARAH-p4Y04p9zdFxHsOPVc8M05IyyUszPPrpQyK_rnZUeTBgHxJcbOgFWmlmXhxSH9mtaYt5OjZW74XLAVSsyGQAZkZMYeo7s-r1wht_AUWc0H1-H7vZX-ZyDlAAFKvaiXXEVdfS94u2mtnli50gRr3K9uAJVpzEzyWs7RXmWE7ow2wFGbTYsArNryTHYrq__BvPUHWbk6MsKFLV08JPDodHq2-hPZvEJYoMGvuI6DFyh1mltU5n4_q-10cv1OhnJZowh_Ciu8w8FxCZxX5QASDKAuYmH0T_xaJTS3sLm2PRuzhpBgbStA4N5rnsGP7LzTDrzspy5r6-Xu8tRe45jrpwJsSq4dwg-OPGh6TYZdoF2htRCy1fUxxfSaCo4SZ3O17yA8G5IhtmI52ju_uLC6kX5BGq7t3ZQDYdP9MrRvnvXTzw4HdHkjVrB_iEM5FpCuFz6cO1IoTL0XF3TdM6_2oJZ8uP17AZPRnEcXhsgIsXiqjrLVvl02NhfqP14uOxuxCxAXkioyxoRxzmssyVHwmeuART7YTV0vTvRmiMwBwIhCcflij-DQDhV99nlk2qu0wfmby529CiIgWtqFWLfaSVNCSlICDoX2rQa7y5vddXUT7Dbo6Pii9W8JLYW0mqph6MsRhedTfDnnpuXXnBrbHNqdrtcC2ouP5yNhlc3ERcgCDU-fGVWGFRYFgx4zNGCuqlfaChJ6JnBj9iHmhdPkIP-54XXHYhvpCT3UxK9Pik98vZJP-DAuUvnHg0Q